In [1]:
import sys
sys.path.append(r"C:\Users\hjia9\Documents\GitHub\data-analysis")
sys.path.append(r"C:\Users\hjia9\Documents\GitHub\data-analysis\ucla-lapd")
import os
import numpy as np
import matplotlib.pyplot as plt
import copy
from scipy.ndimage import gaussian_filter1d, gaussian_filter
from scipy.signal import find_peaks, savgol_filter

import read_hdf5 as rh
from bapsflib import lapd

from lp_analysis import find_sweep_indices, reshape_IV
from lp_iv_analysis import analyze_IV


%matplotlib qt
plt.rcParams.update({'font.size': 16})

In [2]:
from turtle import pos


ifn = r"D:\data\LAPD\Mar26-data\10-emiss-xyplane-coarse-bias.hdf5"

# Extract directory and run number from ifn
data_dir = os.path.dirname(ifn)
run_num = os.path.basename(ifn).split('-')[0]  # "11" from "11-lp-sweep-xyplane-bias.hdf5"
# Create output filename
save_path = os.path.join(data_dir, f"{run_num}-sweep-data.npz")

with lapd.File(ifn) as f:
    rh.show_info(f)
    adc, digi_dict = rh.read_digitizer_config(f)
    pos_dict, xpos, ypos, zpos, npos, nshot = rh.read_bmotion_probe_motion(f)
    key = list(pos_dict.keys())[0]
    pos_array = pos_dict[key]

c:\Users\hjia9\Documents\GitHub\data-analysis\.venv\Lib\site-packages\bapsflib-0.0.0-py3.11.egg\bapsflib\_hdf\maps\digitizers\siscrate.py:728: UserWarning: HDF5 structure unexpected...'siscf-1ch-emissive/SIS crate 3302 configurations[0]' does not define any valid channel numbers...not adding to `configs` dict
  warn(why)


10-emiss-xyplane-coarse-bias.hdf5 Overview
Generated by bapsflib (v0.0.0)
Generated date: 3/31/2026 11:03:12 PM


Filename:     10-emiss-xyplane-coarse-bias.hdf5
Abs. Path:    D:\data\LAPD\Mar26-data\10-emiss-xyplane-coarse-bias.hdf5
LaPD version: 1.2
Investigator: Jia Han
Run Date:     3/11/2026 3:29:50 PM

Exp. and Run Structure:
  (set)  Electrode_Biasing
  (exp)  +-- mar2026-jia
  (run)  |   +-- 10-emiss-xyplane-coarse-bias

Run Description:
    Recording plasma potential from moving emissive probe (P30) measurements on a coarse plane with bias; same condition as run09.
    Most diagnostic and bias settings same as feb2026 experiments.
    
    
    LAPD B field:
    Black magnets at south: 888 A (PS12, 13),
    Yellow & Purple magnets: configured for uniform 800 G
    Except Yellow magnet PS 3, 4: 0 kG
    Black magnets at north: 0 A (PS11)
    (1600G-Source: 800 G: Bulk, 0 G: north end)
    
    South LaB6 source:
    He plasma, Discharge PS Voltage: 80 V, discharge current: 4.3 

In [3]:
with lapd.File(ifn) as f:
    data, tarr = rh.read_data(f, 4, 8, index_arr=slice(npos*nshot), adc=adc)
    Vp_arr = data['signal'].reshape((npos, nshot, -1)) * 40

c:\Users\hjia9\Documents\GitHub\data-analysis\.venv\Lib\site-packages\bapsflib-0.0.0-py3.11.egg\bapsflib\_hdf\maps\digitizers\siscrate.py:728: UserWarning: HDF5 structure unexpected...'siscf-1ch-emissive/SIS crate 3302 configurations[0]' does not define any valid channel numbers...not adding to `configs` dict
  warn(why)
c:\Users\hjia9\Documents\GitHub\data-analysis\.venv\Lib\site-packages\bapsflib-0.0.0-py3.11.egg\bapsflib\_hdf\utils\hdfreaddata.py:508: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  data = np.empty(shape, dtype=dtype)


In [7]:
Vp_arr_avg = np.mean(Vp_arr, axis=1)  # Average over shots

In [ ]:
plt.figure()
for i in [0, 100, 200, 300, 400]:
    plt.plot(tarr, Vp_arr[i, 0], label=pos_array[i])
plt.legend()
plt.xlabel("Time")
plt.ylabel("Voltage")
plt.title("Plasma potential")
plt.show()

In [28]:
extent = min(xpos), max(xpos), min(ypos), max(ypos)
grid_shape = len(np.unique(xpos)), len(np.unique(ypos))

tndx_list = [16500, 23000, 38500, 82500]

# Set up a figure with 4 subplots
fig, axs = plt.subplots(2, 2, figsize=(16, 12))

for i, tndx in enumerate(tndx_list):
    t = tarr[tndx] * 1e3 + 14 - 15.5
    # Average over +-100 indices around tndx
    Vp_avg = np.mean(Vp_arr_avg[:, tndx-100:tndx+101], axis=1)
    # Extract the data for the requested sweep and reshape to 2D
    Vp_2d = Vp_avg.reshape(grid_shape)
    
    ax = axs.flat[i]
    ax.set_title(f't = {t:.2f} ms')
    im = ax.imshow(Vp_2d, origin='lower', cmap=plt.cm.rainbow, extent=extent, interpolation='gaussian', vmin=-5, vmax=25)
    ax.set_xlabel('X Position')
    ax.set_ylabel('Y Position')
    ax.set_aspect('equal')

# Add a single colorbar for all subplots
cbar_ax = fig.add_axes([0.92, 0.1, 0.02, 0.8])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label('Plasma Potential (V)')

plt.tight_layout()
plt.show()

C:\Users\hjia9\AppData\Local\Temp\ipykernel_25652\2385207691.py:28: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [6]:
Vp_arr.shape

(441, 4, 108544)